**Uczenie Głębokie (laboratorium)**

Dominik Filipiak

`df[🏒]amu.edu.pl`

Materiały zawierają wybrane fragmenty materiałów pomocniczych, które przygotowałem na potrzeby kursu Deep Learning dla studentów kierunku Machine Learning na Wydziale Matematyki, Informatyki i Mechaniki UW.

Cześć kodu bazuje na [notebookach z kursu Deep Learning](https://github.com/mim-ml-teaching/public-dnn-2024-25) dla studentów kierunku Machine Learning na Wydziale Matematyki, Informatyki i Mechaniki UW.

# Wprowadzenie do Computer Vision

## Semantyczna segmentacja

Semantyczna segmentacja to zadanie przypisania etykiety (klasy) każdemu pikselowi obrazu.
W odróżnieniu od klasyfikacji (poziom obrazu) czy detekcji (poziom instancji obiektu), segmentacja klasyfikuje obraz na poziomie pikselowym.

### Definicja problemu

**Segmentacja semantyczna** (ang. *semantic segmentation*) to zadanie przetwarzania obrazu, którego celem jest przypisanie każdemu pikselowi jednej z predefiniowanych klas (tworząc tym samym spójne semantycznie obszary).

W odróżnieniu od klasyfikacji obrazów (gdzie przypisujemy etykietę całemu obrazowi) czy detekcji obiektów (gdzie lokalizujemy i klasyfikujemy obiekty), segmentacja semantyczna skupia się na **precyzyjnym oznaczaniu granic obiektów** w obrazie.

Formalnie, dla danego obrazu wejściowego $x \in \mathbb{R}^{H \times W \times 3}$ (zakładamy 3 kanały RGB) nasz model $f$ (z parametrami $\theta$) ma za zadanie przewidzieć mapę klas (zwaną często maską) $y \in \{0, \ldots, C-1\}^{H \times W}$, gdzie `C` to liczba klas.

Podsumowując, nasz model definiujemy następująco:
$$f_{\theta}\colon \mathbb{R}^{H \times W \times 3} \mapsto \{0, \ldots, C-1\}^{H \times W}$$

<details>
  <summary>Przykłady zasosowań</summary>

- segmentacja struktur anatomicznych na obrazach medycznych (np. MRI),
- analiza zdjęć satelitarnych,
- segmentacja drogi i przeszkód w autonomicznych pojazdach,
- oddzielanie człowieka od tła podczas telekonferencji na Teamsach (celem rozmycia tła).
</details>

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np

import shutil
from datetime import datetime
from pathlib import Path
from typing import Callable, Dict, Optional, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from tqdm import tqdm

matplotlib.style.use('dark_background')


In [ ]:

activation_maps: Dict[str, torch.Tensor] = {}

def get_torch_device() -> torch.device:
    """Funkcja pomocnicza do określenia na którym urządzeniu będziemy wykonywać obliczenia."""
    if torch.cuda.is_available():
        return torch.device("cuda") # NVIDIA GPU
    elif torch.backends.mps.is_available():
        return torch.device("mps")  # Apple Silicon - uwaga, nie wszysko działa. W razie kłopotów przejdź na CPU.
    else:
        return torch.device("cpu")  # CPU

def get_activation(name: str) -> Callable:
    """
    Zwraca hook do przechwytywania aktywacji dla danej warstwy.

    Args:
        name: unikalna nazwa warstwy (z `named_modules()`)

    Returns:
        Funkcja hooka do rejestrowania aktywacji.
    """
    def hook(model: nn.Module, input: torch.Tensor, output: torch.Tensor) -> None:
        activation_maps[name] = output.detach().cpu()
    return hook

def register_hooks(model: nn.Module) -> None:
    """
    Rejestruje forward hooki dla wszystkich warstw nn.Conv2d w modelu.

    Args:
        model: model sieci neuronowej
    """
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            module.register_forward_hook(get_activation(name))

def log_activation_maps(writer: SummaryWriter, model: nn.Module, input_img: torch.Tensor, epoch: int, device: torch.device) -> None:
    """
    Loguje mapy aktywacji do TensorBoard dla wszystkich warstw konwolucyjnych.

    Args:
        writer: obiekt SummaryWriter
        model: model z zarejestrowanymi hookami
        input_img: obraz wejściowy (tensor o wymiarach C x H x W)
        epoch: numer epoki (do logowania)
        device: urządzenie, na którym znajduje się model
    """
    model.eval()
    with torch.no_grad():
        model(input_img.unsqueeze(0).to(device))

    for name, activation in activation_maps.items():
        if activation.dim() == 4:
            # Batch x Channels x H x W → pokazujemy maks. 8 kanałów
            img_grid = torchvision.utils.make_grid(
                activation[0, :16].unsqueeze(1), normalize=True, scale_each=True
            )
            writer.add_image(f"Activations/{name}", img_grid, epoch)

def log_filters(writer: SummaryWriter, model: nn.Module, epoch: int) -> None:
    """
    Loguje wizualizacje filtrów wszystkich warstw nn.Conv2d do TensorBoard.

    Args:
        writer: obiekt SummaryWriter
        model: model sieci neuronowej
        epoch: numer epoki (do logowania)
    """
    conv_idx = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            weights = module.weight.data.clone()

            # Normalizacja do [0, 1]
            weights = (weights - weights.min()) / (weights.max() - weights.min() + 1e-5)

            if weights.shape[1] in {1, 3}:
                writer.add_images(f"{name}/Conv{conv_idx}-filters", weights, epoch)
            else:
                writer.add_images(f"{name}/Conv{conv_idx}-filters_in0", weights[:, 0:1, :, :], epoch)

            conv_idx += 1        

def log_weights_and_gradients(writer: SummaryWriter, model: nn.Module, epoch: int) -> None:
    """
    Loguje histogramy wag, biasów i ich gradientów dla wszystkich modułów modelu.

    Args:
        writer: obiekt SummaryWriter
        model: model sieci neuronowej
        epoch: numer epoki (do logowania)
    """
    for name, module in model.named_modules():
        if hasattr(module, "weight") and module.weight is not None:
            writer.add_histogram(f"Weights/{name}/weight", module.weight.data.cpu(), epoch)
            if module.weight.grad is not None:
                writer.add_histogram(f"Gradients/{name}/weight", module.weight.grad.cpu(), epoch)
        
        if hasattr(module, "bias") and module.bias is not None:
            writer.add_histogram(f"Weights/{name}/bias", module.bias.data.cpu(), epoch)
            if module.bias.grad is not None:
                writer.add_histogram(f"Gradients/{name}/bias", module.bias.grad.cpu(), epoch)

def count_params(model: nn.Module) -> int:
    """
    Zlicza liczbę parametrów w modelu.

    Args:
        model: instancja nn.Module

    Returns:
        Całkowita liczba parametrów w modelu.
    """
    return sum(p.numel() for p in model.parameters())

# tu drobne zmiany na potrzeby segmentacji
def log_predictions(writer: SummaryWriter, model: nn.Module, dataset, epoch: int, device: torch.device) -> None:
    model.eval()
    images = []
    targets = []
    preds = []

    with torch.no_grad():
        for i in range(4):
            x, y = dataset[i]
            x = x.to(device).unsqueeze(0)
            y = y.numpy()
            output = model(x)
            pred = output.argmax(dim=1).squeeze().cpu().numpy()

            images.append(x.squeeze(0).cpu())
            targets.append(y)
            preds.append(pred)

    fig, axs = plt.subplots(4, 3, figsize=(10, 10))
    for i in range(4):
        axs[i, 0].imshow(images[i].permute(1, 2, 0))
        axs[i, 0].set_title("Wejście")
        axs[i, 1].imshow(images[i].permute(1, 2, 0))
        axs[i, 1].imshow(targets[i], cmap="viridis", alpha=0.5)
        axs[i, 1].set_title("Maska GT")
        axs[i, 2].imshow(images[i].permute(1, 2, 0))
        axs[i, 2].imshow(preds[i], cmap="viridis", alpha=0.5)
        axs[i, 2].set_title("Predykcja")
        for j in range(3):
            axs[i, j].axis("off")
    plt.tight_layout()
    writer.add_figure("Predictions/SemanticSegmentation", fig, global_step=epoch)
    plt.close(fig)

### Metryka ewaluacyjna

#### IoU (Intersection over Union)

Jedną z kluczowych metryk używanych do oceny jakości segmentacji jest **IoU** (ang. *Intersection over Union*), znana też jako **Jaccard Index**.
Dla danej klasy $c$:
$$
\text{IoU}_c = \frac{|\text{Prediction}_c \cap \text{GroundTruth}_c|}{|\text{Prediction}_c \cup \text{GroundTruth}_c|}
$$

- Licznik: liczba pikseli poprawnie zaklasyfikowanych jako klasa $c$,
- Mianownik: łączna liczba pikseli przewidzianych lub rzeczywistych jako klasa $c$.

#### mIoU
**Mean IoU** (mIoU) to średnia wartość IoU po wszystkich klasach (z pominięciem klas nieobecnych).

<details>
  <summary>Wizualizacja IoU</summary>

![Source: wikimedia](https://upload.wikimedia.org/wikipedia/commons/c/c7/Intersection_over_Union_-_visual_equation.png)

</details>

<details>
  <summary>Dlaczego nie accuracy?</summary>

Accuracy może być złudnie wysokie przy klasach dominujących (np. tło, co jest typowe dla segmentacji).

Załóżmy, że na obrazie 100 $\times$ 100 jest 10 000 pikseli:
- 9 800 to tło (klasa 0),
- 200 to obiekt (klasa 1).

Załóżmy, że nasz model zgaduje tło wszędzie - $f_{\theta}(x) = 0$.
- $\text{Accuracy} = 0.98$ (bo 9 800 z 10 000 pikseli to tło),
- $\text{IoU}_{c=0.98} = 0$ i $\text{IoU}_{c=1} = 0$, więc $\text{mIoU} = \frac{0.98 + 0}{2} = 0.49$.
</details>


In [ ]:
def compute_iou(pred: torch.Tensor, target: torch.Tensor, c: int) -> float:
    """Oblicza IoU dla klasy `c`.

    Args:
        pred (torch.Tensor): Tensor z naszymi predykcjami
        target (torch.Tensor): Tensor z prawdziwymi maskami 
        c (int): Klasa dla której liczymy IoU

    Returns:
        float: IoU dla klasy `c`
    """
    prediction_c = (pred == c)
    ground_truth_c = (target == c)
    
    intersection = (prediction_c & ground_truth_c).sum().item()   # licznik
    union = (prediction_c | ground_truth_c).sum().item()          # mianownik

    if union == 0:
        return float('nan')
    else:
        return intersection / union

def compute_miou(pred: torch.Tensor, target: torch.Tensor, num_classes: int) -> float:
    """Oblicza mIoU.

    Args:
        pred (torch.Tensor): Tensor z naszymi predykcjami
        target (torch.Tensor): Tensor z prawdziwymi maskami 
        num_classes (int): Liczba wszystkich klas

    Returns:
        float: mIoU.
    """
    ious = [compute_iou(pred, target, c) for c in range(num_classes)]
    return np.nanmean(ious)

In [ ]:
def train_epoch(
    model: nn.Module,
    device: torch.device,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    writer: Optional[SummaryWriter] = None,
    log_histograms: bool = False
) -> Tuple[float, float]:
    """
    Trenuje model przez jedną epokę na zadanym zbiorze danych.

    Args:
        model: Model sieci neuronowej (w trybie treningowym).
        device: Urządzenie obliczeniowe (CPU / CUDA / MPS).
        loader: DataLoader zawierający dane treningowe.
        optimizer: Obiekt optymalizatora.
        epoch: Numer epoki (do logowania).
        writer: Obiekt SummaryWriter do logowania w TensorBoard.
        log_histograms: Czy logować histogramy wag i gradientów.

    Returns:
        Tuple zawierający średnią wartość funkcji straty oraz dokładność dla epoki.
    """
    model.train()
    total_loss = 0
    correct = 0

    for batch_idx, (data, target) in enumerate(tqdm(loader, desc=f"Train Epoch {epoch}")):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = nn.CrossEntropyLoss()(output, target)
        loss.backward()

        if log_histograms and writer:
            log_weights_and_gradients(writer, model, epoch)

        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()

    avg_loss = total_loss / len(loader)

    if writer:
        writer.add_scalar("Train/Loss", avg_loss, epoch)
        example_input = loader.dataset[0][0].to(device)
        log_activation_maps(writer, model, example_input, epoch, device=device)
        log_filters(writer, model, epoch)

    return avg_loss, None

def val_epoch(
    model: nn.Module,
    device: torch.device,
    loader: DataLoader,
    epoch: int,
    writer: Optional[SummaryWriter] = None,
    dataset = None,
    num_classes=3,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0
    correct = 0
    iou_scores = []

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = nn.CrossEntropyLoss()(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            iou_scores.append(compute_miou(pred.cpu(), target.cpu(), num_classes))
            

    avg_loss = total_loss / len(loader)
    mean_iou = np.nanmean(iou_scores)

    if writer:
        writer.add_scalar("Val/Loss", avg_loss, epoch)
        writer.add_scalar("Val/mIoU", mean_iou, epoch)
        log_predictions(writer, model, loader.dataset, epoch, device)

    return avg_loss, mean_iou

def run_training(
    runs_dir: Path,
    device: torch.device,
    epochs: int,
    learning_rate: float,
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader
) -> None:
    """
    Uruchamia pełen proces treningu i walidacji modelu.

    Args:
        runs_dir: Ścieżka do katalogu, gdzie zapisywane będą logi TensorBoard.
        device: Urządzenie (CPU / CUDA / MPS).
        epochs: Liczba epok do przeprowadzenia treningu.
        learning_rate: Współczynnik uczenia.
        model: Model do trenowania (torch.nn.Module).
        train_loader: Dane treningowe.
        test_loader: Dane testowe lub walidacyjne.
    """
    register_hooks(model)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    run_name = f"{model.__class__.__name__}_{str(datetime.now().timestamp())}"

    best_val_loss = torch.inf

    with SummaryWriter(runs_dir / run_name) as writer:
        writer.add_graph(model, train_loader.dataset[0][0].unsqueeze(0).to(device))

        for epoch in range(epochs):
            train_epoch(model, device, train_loader, optimizer, epoch, writer)
            val_loss, _ = val_epoch(model, device, test_loader, epoch, writer)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state_dict = model.state_dict()

        model.load_state_dict(best_state_dict)
        test_loss, test_acc = val_epoch(model, device, test_loader, epoch + 1, writer)

        writer.add_hparams(
            {
                "model": model.__class__.__name__,
                "lr": learning_rate,
                "epochs": epochs,
            },
            {
                "hparam/num_params": count_params(model),
                "hparam/test_loss": test_loss,
                "hparam/test_accuracy": test_acc
            },
            run_name=run_name
        )


### Splot transponowany (`ConvTranspose2d`)

- rekonstruuje strukturę lokalną z reprezentacji cech i zwiększa rozdzielczość,
- nie jest dosłowną matematyczną odwrotnością `Conv2d`, ale działa tak, by rekonstruować przestrzeń.

<details>
  <summary>Wizualizacja</summary>

![Source: Zefs guides](figures/tconv.png)

</details>


### Architektura UNet

UNet to sieć konwolucyjna zaprojektowana pierwotnie do segmentacji obrazów medycznych. Składa się z dwóch części:
- **Encoder**: stopniowo zmniejsza rozmiary obrazu, ucząc się cech na różnych poziomach (w praktyce w tej roli często jest ResNet/VGG jako backbone).
- **Decoder**: stopniowo zwiększa rozdzielczość, tworząc mapę segmentacyjną.
- **Skip connections**: łączą warstwy encodera z odpowiadającymi im warstwami decodera.

Więcej przeczytasz w [oryginalnym artykule](https://arxiv.org/pdf/1505.04597).

In [ ]:
def create_unet_block(in_c: int, out_c: int) -> nn.Sequential:
    """Blok 2x(`Conv2d` -> `BatchNorm2d` -> `ReLU`)"""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super(UNet, self).__init__()

        # encoder
        self.enc1 = create_unet_block(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = create_unet_block(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = create_unet_block(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = create_unet_block(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # bottleneck
        self.bottleneck = create_unet_block(512, 1024)

        # decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = create_unet_block(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = create_unet_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = create_unet_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = create_unet_block(128, 64)

        # final conv 1x1
        self.conv_last = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool1(enc1))
        enc3 = self.enc3(self.pool2(enc2))
        enc4 = self.enc4(self.pool3(enc3))

        # bottleneck
        bottleneck = self.bottleneck(self.pool4(enc4))

        # decoder
        dec4 = self.dec4(torch.cat((self.upconv4(bottleneck), enc4), dim=1))
        dec3 = self.dec3(torch.cat((self.upconv3(dec4), enc3), dim=1))
        dec2 = self.dec2(torch.cat((self.upconv2(dec3), enc2), dim=1))
        dec1 = self.dec1(torch.cat((self.upconv1(dec2), enc1), dim=1))

        # final conv 1x1
        return self.conv_last(dec1)

In [ ]:
from torchvision.datasets import OxfordIIITPet

IMG_SIZE = (128, 128)

class OxfordPetDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="train", transform=None, target_transform=None):
        self.dataset = OxfordIIITPet(
            root=root,
            split=split,
            target_types="segmentation",
            download=True
        )
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, mask = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
        return image, mask


transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

def squeeze_and_long(tensor):
    return tensor.squeeze().long()

target_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
    transforms.Lambda(squeeze_and_long)
])

data_root = "./data"

train_dataset = OxfordPetDataset(data_root, split="trainval", transform=transform, target_transform=target_transform)
test_dataset = OxfordPetDataset(data_root, split="test", transform=transform, target_transform=target_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

In [ ]:
# uwaga - usuwa poprzednie eksperymenty!

runs_dir = Path("runs")
shutil.rmtree(runs_dir)
runs_dir.mkdir(exist_ok=True)

device = get_torch_device()
model = UNet(in_channels=3, out_channels=3).to(device)  # 3 klasy: tło, pies, kot


run_training(
    runs_dir=runs_dir,
    device=device,
    epochs=15,
    learning_rate=1e-3,
    model=model,
    train_loader=train_loader,
    test_loader=test_loader
)


In [ ]:
# Do zapisu
torch.save(model.state_dict(), "unet_epoch-14.pth")

# Do wczytania
# model = torch.load("unet_epoch9.pth", weights_only=False)

In [ ]:
def visualize_sample(model, dataset, device, idx=0):
    model.eval()
    image, mask = dataset[idx]
    with torch.no_grad():
        input_tensor = image.unsqueeze(0).to(device)
        output = model(input_tensor)
        pred = output.argmax(dim=1).squeeze().cpu().numpy()

    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask.numpy()

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(image_np)
    axs[0].set_title("Wejście")
    axs[1].imshow(mask_np, cmap="gray")
    axs[1].set_title("Maska GT")
    axs[2].imshow(pred, cmap="gray")
    axs[2].set_title("Predykcja")
    for ax in axs:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


visualize_sample(model, train_dataset, device, idx=3000)

#### Bonus - profilowanie

In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

image, mask = test_dataset[0]
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA, ProfilerActivity.XPU],
    record_shapes=True,
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/resnet18'),
    with_stack=True
    ) as prof:
    with record_function("model_inference"):
        model(image.unsqueeze(0).to(device))

    print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

# Autoenkodery – wprowadzenie do generetywnych sieci.

## Definicja

**Autoenkoder** to rodzaj sieci neuronowej uczonej w sposób nienadzorowany, której celem jest odwzorowanie wejścia na wyjście przy jednoczesnym kompresowaniu informacji.

Składa się z dwóch głównych części:
- **Encoder** – przekształca dane wejściowe w reprezentację latentną o niższym wymiarze (cechy ukryte),
- **Decoder** – rekonstruuje dane wejściowe na podstawie tej reprezentacji.

Formalnie:
- Dla wejścia $ x \in \mathbb{R}^n $,
- Encoder: $ z = f(x; \theta) \in \mathbb{R}^k $, gdzie $ k \ll n $,
- Decoder: $ \hat{x} = g(z; \phi) \approx x $.

Uczy się przez minimalizację funkcji straty rekonstrukcji, np. MSE:
$$
\mathcal{L}_{\text{AE}}(x, \hat{x}) = \|x - \hat{x}\|^2
$$

### Autoenkoder konwolucyjny

W przypadku danych obrazowych stosuje się **konwolucyjne autoenkodery**, gdzie:
- Encoder to ciąg warstw `Conv2d + ReLU + MaxPool`,
- Decoder to ciąg warstw `ConvTranspose2d + ReLU`.

Autoenkoder nie tylko kompresuje dane, ale również **uczy się ekstrakcji cech przestrzennych**.

### Związek z UNet

UNet można traktować jako **autoenkoder rozszerzony o skip connections**:
- Część „downsamplingowa” UNet to klasyczny encoder,
- Część „upsamplingowa” pełni funkcję dekodera,
- Dodatkowe połączenia boczne (skip connections) przekazują lokalne informacje przestrzenne z encodera do decodera.

Celem nie jest rekonstrukcja obrazu wejściowego, lecz przewidywanie **maski segmentacyjnej**, czyli klasyfikacja każdego piksela.

### Zastosowania autoenkoderów

- Redukcja wymiarowości (w pewnym sensie alternatywa dla PCA!),
- Anomaly detection – rekonstrukcja jako kryterium „normalności” (choć są lepsze metody),
- Wstępne uczenie reprezentacji do downstream tasks (np. segmentacja, klasyfikacja),
- kolorowanie zdjęć,
- syntetyczne dane (wykorzystywany w [GAN](https://arxiv.org/pdf/1406.2661), [VAE](https://arxiv.org/pdf/1312.6114), [Stable Diffusion](https://arxiv.org/pdf/2006.11239)...).


<details>
  <summary>Wizualizacja 1</summary>

![Source: edureka.co](https://d1jnx9ba8s6j9r.cloudfront.net/blog/wp-content/uploads/2018/10/1_8ixTe1VHLsmKB3AquWdxpQ.png)
</details>

<details>
  <summary>Wizualizacja 2</summary>

![Source: edureka.co](https://d1jnx9ba8s6j9r.cloudfront.net/blog/wp-content/uploads/2018/10/1_G0V4dz4RKTKGpebeoSWB0A.png)

</details>

<details>
  <summary>Wizualizacja 3</summary>

![Source: edureka.co](https://d1jnx9ba8s6j9r.cloudfront.net/blog/wp-content/uploads/2018/10/1_has2O8b3HAUqvcqqLrlBQA.png)

</details>

## AE w praktyce

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.ReLU(True)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


def train_autoencoder(model, train_loader, device, epochs=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, _ in tqdm(train_loader):
            imgs = imgs.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, imgs)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss / len(train_loader):.4f}")


In [ ]:
autoencoder = ConvAutoencoder()

train_autoencoder(autoencoder, train_loader, device, epochs=10)

In [ ]:

def visualize_autoencoder(model: nn.Module, dataset, device: torch.device, idx: int = 0):
    model.eval()
    with torch.no_grad():
        x, _ = dataset[idx]
        x_input = x.unsqueeze(0).to(device)
        z = model.encoder(x_input)
        x_recon = model.decoder(z)

    x_np = x.permute(1, 2, 0).cpu().numpy()
    z_np = torch.mean(z.squeeze(0), dim=0).cpu().numpy()  # średnia mapa cech
    x_recon_np = x_recon.squeeze(0).permute(1, 2, 0).cpu().numpy()

    fig, axs = plt.subplots(1, 4, figsize=(16, 4))
    axs[0].imshow(x_np)
    axs[0].set_title("Wejście")

    axs[1].imshow(z_np, cmap="viridis")
    axs[1].set_title("Reprezentacja z")

    axs[2].imshow(x_recon_np)
    axs[2].set_title("Rekonstrukcja")

    for ax in axs:
        ax.axis("off")

    error_map = np.abs(x_np - x_recon_np).mean(axis=2)
    axs[3].imshow(error_map, cmap="inferno")
    axs[3].set_title("Błąd |x−x̂|")
    axs[3].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_autoencoder(autoencoder, test_dataset, device, idx=1)

# Zadania do samodzielnego rozwiazania

## Z1.

Do tej pory monitorowaliśmy tylko mIoU. Napisz kod, który będzie logował IoU dla kazdej klasy do Tensorboard. Jako bazę wykorzystaj kod z zajeć.


In [ ]:
def val_epoch(
    model: nn.Module,
    device: torch.device,
    loader: DataLoader,
    epoch: int,
    writer: Optional[SummaryWriter] = None,
    dataset = None,
    num_classes=3,
) -> Tuple[float, float]:
    model.eval()
    total_loss = 0
    correct = 0
    miou_scores = []
    iou_scores_per_class = {c: [] for c in range(num_classes)}

    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = nn.CrossEntropyLoss()(output, target)
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            miou_scores.append(compute_miou(pred.cpu(), target.cpu(), num_classes))
            for c in range(num_classes):
                iou_scores_per_class[c].append(compute_iou(pred, target, c))


    avg_loss = total_loss / len(loader)
    mean_iou = np.nanmean(miou_scores)

    iou_scores_per_class = torch.tensor([np.nanmean(iou_scores_per_class[c]) for c in range(num_classes)])

    if writer:
        writer.add_scalar("Val/Loss", avg_loss, epoch)
        writer.add_scalar("Val/mIoU", mean_iou, epoch)
        writer.add_tensor("Val/ious", iou_scores_per_class, epoch)
        log_predictions(writer, model, loader.dataset, epoch, device)

    return avg_loss, mean_iou

## Z2.
Dodaj rozszerz przykład U-net o rotację (np. `RandomHorizontalFlip` / `RandomRotation`). Zbadaj wpływ na wyniki czy poprawia wyniki. Pamiętaj, ze obracamy zarówno obrazki, jaki targety!

In [ ]:
import random
import torchvision.transforms.functional as TF

In [ ]:
class CommonHFlipTransform:

    def __init__(self, flip_prob: float, max_deg: float):
        self.flip_prob = flip_prob
        self.rotate = transforms.RandomRotation(degrees=max_deg)

    def __call__(self, img, mask):
        if random.random() < self.flip_prob:
            img = TF.hflip(img)
            mask = TF.hflip(mask)

        angle = self.rotate.get_params(self.rotate.degrees)
        img = TF.rotate(img, angle)
        mask = TF.rotate(mask, angle)

        return img, mask

In [ ]:
from torchvision.datasets import OxfordIIITPet

IMG_SIZE = (128, 128)

class OxfordPetDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="train", common_transform=None,
                 transform=None, target_transform=None):
        self.dataset = OxfordIIITPet(
            root=root,
            split=split,
            target_types="segmentation",
            download=True
        )
        self.common_transform = common_transform
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, mask = self.dataset[idx]
        if self.common_transform:
            image, mask = self.common_transform(image, mask)
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)
        return image, mask


common_transform = CommonHFlipTransform(0.5, 30)

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

def squeeze_and_long(tensor):
    return tensor.squeeze().long()

target_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
    transforms.Lambda(squeeze_and_long)
])

data_root = "./data"

train_dataset = OxfordPetDataset(data_root, split="trainval", common_transform=common_transform,
                                 transform=transform, target_transform=target_transform)
test_dataset = OxfordPetDataset(data_root, split="test", common_transform=common_transform,
                                transform=transform, target_transform=target_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)